# Convolution Layer

This notebook provides a comprehensive analysis of what convolution layers are.

The notebook is divided into the following sections:
1. The need for convolution
2. Introduction to convolution functions and convolution neural network
3. Details about convolution neural networks
4. Understanding how convolution operation is implemented in pytorch

**Note**:
- For implementation of how torch.nn.Conv2d works you can look at `pytorch-fundamentals/src/pytorch_fundamentals/layers/cnn.py`

## The need for convolution

There has been a significant advantage in using **learnable weights and backpropagation combination** to learn a given set of data to perform a certain task over using fixed weights that are produced as a process of long hours of feature engineering. The most popular implementation of learnable weights and backprogation combination is the **ANN** layer. The **ANN** (Artificial Neural Network) layer is a layer that presents neural nets in a linear line. The stack of these ANN layers creates a **Deep Neural Network** (DNN). All inputs of an ANN layer are connect to all of the outputs. ANN layer is popularly also known as a **fully connected layers**.

### Operations taking place in a single ANN layer
If the incoming feature is represented as $X$, then ANN uses a set of weights $W$ and bias $B$ to propagate the features through the layer to create an output $y$:<br>
$y = X * W.T + B$ where:<br>
$X$ = A tensor of features of shape $(N, L)$ (number of data-points, number of features representing a single data point).<br>
$W$ = A tensor of learnable weights of shape $(L_{out}, L)$ (number of features required to represent a single datapoint in the output, number of features representing a single data point of the input tensor).<br>
$B$ = A tensor of learnable biases of shape $(L_{out})$ (number of features required to represent a single datapoint in the output).<br>

$W.T$ = (denotes transposed $W$ to allow for matrix multiplication between $X$ and $W$)

### Constraints of an ANN Linear Layer
#### 1. Large number of training weights required to train DNN with multiple ANN layers
Whenever we want to train a neural network model on a set of data to perform a certain task we need to look at how many trainable parameters can fit on our resources. An ANN linear layer is a dense layer, the number of parameters part of the linear layer to be trained increase exponentially when number of features to be trained on increases.
Let's take an example of a single ANN linear layer where we see two cases. 
1. Where we want to go from 100 input features ($L = 100$) to an output of 50 features ($L_{out} = 50$) (halving the input features in 1 layer). 
2.  Where we want to go from 100 input features ($L = 100$) to an output of 100 features ($L_{out} = 50$).

The total learnable weights required will be **5050** (100 x 50 + 50) and **10100** (100 x 100 + 100) respectively. A DNN model needs multiple layers to go from low level features to higher level of features (which contain information about how the low level input features are related to each other) to then use these high level features to train on the task. Multiple ANN layers would thus lead to requirement of a very large set of learnable weights and biases. To train such large amount of parameters we require a dataset with larger number of datapoints that the parameters to generalise well otherwise it would only lead to **overfitting**.

#### 2. Lose information about the original data
An ANN linear layer can only work with all features present in a single layer (**flattened layer**). When it comes to real world problems the data is present in the form of **images**, **audio** and **text** (these are what we call **modal** of data). These modals are not structured as a single linear layer. Images store spatial information about the data, audio and text store temporal information about the data that they represent.
When we convert these images, audio and text data from their original orientation to low level features placed in a linear fashion we lose all of the spatial and temporal information. The only way this information can be retained is when a large amount of time is spent to **feature engineer** these features before passing it to the ANN linear layer and as these modals contain vast amount of information which can not be feature engineered through human efforts. The whole purpose of DNNs is to move away from feature engineering and allow trainable weights across layers to learn these information. 

### Moving to convolution neural network
To overcome the two hurdles of exponential explosion of parameters as we increase the size of deep neural networks and the destruction of structural context as we flatten different modals of data into a flattened linear layer in order to use linear ANN layers **Yann LeCun** who insisted on introducing **convolution functions** to build DNN models on image based tasks.

## Convolution Functions and Convolution Neural Network

Convolution is a mathematical operation between 2 functions **f** and **g** to generate a third function. In this convolution operation one function modifies the shape of the second function. The convolution function that modifies is called the **convolution kernel**. A set of **convolution kernel** is called **convolution filter**.

The convolution filter convolves (slides) over the image and generates a third function called a **feature map**. Each element on the feature map is **the sum of product between the receptive field of the original input and the kernels of the convolution filter. To each element on the feature map a bias is also added**. The feature maps generated due to the convolution filter will never be larger than the original input in terms of (height and width).

### Advantages of using convolution filters
1. Images have a strong local structure where spatially close pixels are highly correlated (neighboring pixels are heavily dependent on one another to form structural meaning (like edges, textures, and boundaries)) and Convolutional filters preserve this structure isolating and extracting localized features efficiently without losing their spatial context.
2. In a dense linear layer every single connection requires a unique weight. In a convolutional layer the same kernel weights are reused across every spatial position of the input. If a feature (such as an edge) shifts position in the input image, its corresponding representation will shift identically in the output feature map.

### Convolution filter Hyper-parameters
When a convolutional layer processes an input we configure exactly how the filters used for the convolution process behaves using four main hyperparameters:
1. **Kernel Size** ($kernelsize$): The spatial dimensions of the filter that slides across the input data. A convolution filter is usually a square filter of size 3x3, 5x5, 7x7, 11x11 etc. The kernel size determines the network's receptive field i.e. **how much local context of the original image does the filter considers at one time**. A smaller kernel (example 3x3) captures fine-grained highly local features (example sharp edges) while a larger kernel (examplee 7x7) captures broader global patterns at the cost of missing tiny details.

2. **Stride** ($stride$): The step size the filter takes as it slides across the input. A stride of 1 means the filter moves one pixel at a time. A stride of 2 means it skips a pixel moving two steps at a time. Stride directly controls the size of the output feature map. Increasing the stride reduces the width and height of the feature map making the network more computationally efficient and summarizing the features into a smaller grid.

3. **Padding** ($padding$): The process of symmetrically adding extra rows and columns of artificial pixels around the outer edges of the input data before applying the filter. Without padding the filter cannot extract information stored at the edges of an image (edge features are under-represented) Padding solves this. The most common padding is **zero padding** what this padding does it it allows the filter to center on the edge pixels of an image at the same time adds zero information for the convolution filter to train on leading to incorporation of edges properly without adding any information which is not related to the image at all

4. **Dilation** ($dilation$): The spacing between the individual values inside the kernel itself. A standard filter has a dilation rate of 1 (all cells are touching). A dilation rate of 2 means there is a one-pixel gap between every element in the kernel, spreading the filter out over a wider area. Dilation artificially expands the receptive field without increasing the number of trainable parameters or losing resolution. This is incredibly useful in tasks like image segmentation or audio processing where the network needs to look at a broad context to understand the data.


## Details about Convolution Neural Networks
**Note**:
1. Keep in mind that in DNN model training the dataset is provided in the form of batches and is commonly denoted as $N$.
2. An n-dimensional matrix is popularly called a **tensor**. (For example: An image is 3 channel H x W matrix which is also a tensor)
3. The notation to represent N images in neural network world is $(N,C,H,W)$.

### Important notations and equations used in convolution
#### 1. Input Tensor, Output Tensor and Convolution Filter dimensions
The shape of the input tensor (a batch of images) to a convolution layer is $(N,C_{in},H_{in},W_{in})$. 
Where:<br>
**N** = Batch size<br>
$C_{in}$ = Number of channels of the input tensor<br>
$H_{in}$ = Height of the matrix representing the input tensor<br>
$W_{in}$ = Width of the matrix representing the input tensor

The shape of the output tensor (feature map) from a convolution layer is $(N,C_{out},H_{out},W_{out})$. 
Where:<br>
**N** = Batch size<br>
$C_{out}$ = Number of channels to be present on the output tensor<br>
$H_{out}$ = Height of the matrix representing the output tensor<br>
$W_{out}$ = Width of the matrix representing the output tensor

The shape of the convolution filter is $(C_{out},C_{in},kernelsize,kernelsize)$
Where:<br>
**N** = Batch size<br>
$C_{out}$ = Number of channels to be present on the output tensor<br>
$C_{in}$ = Number of channels of the input tensor<br>
$kernelsize$ = Height and width of a convolution kernel present in a convolution filter 

The shape of the bias as part of the convolution operation is $C_out$
Where:<br>
$C_{out}$ = Number of channels to be present on the output tensor<br>


#### 2. Details on how convolution operation takes place
An image in the real world is generally in the form of a **RGB** image. Most of the image based models are trained on square images of the shape **224 x 224 (H x W)**. So a single RGB image of dimension 224 x 224 pixles is a **3 channel 224 x 224 image**. If i have a batch of 10 images of dimensions 224 x 224 we then denote it's shape/dimension as $(10,3,224,224)$ (input tensor).

If a linear ANN layers was used to process this batch of images then **every element or neuron present in a neural network layer contributes to every element present in the output**, but incase of convolution layers only the receptive field per channel required by the convolution filter contributes to it's output element in the feature map.

We determine the shape of the feature map based on the size of the kernel (kernel size) sliding over the original image, what is the kernel's sliding step size (stride), how well we want the edges to contribute information to the feature map (padding) and how many pixels we want to skip within the receptive field (dilation). $H_{out}$ and $W_{out}$ are determined using the following 2 equations

#### 3. Equations to calculate a feature map's dimension
For feature map's height we use
$$H_{out} = \lfloor(\frac{H_{in} + (2 * padding) - (dilation * (kernelsize - 1) - 1)}{stride})\rfloor - 1$$
For feature map's width we use
$$W_{out} = \lfloor(\frac{W_{in} + (2 * padding) - (dilation * (kernelsize - 1) - 1)}{stride})\rfloor - 1$$
$\lfloor \rfloor$ denote mathematical floor operation <br>

We now know that only the receptive field from each input channel and their product with the convolution filter (with addition of bias) is responsible for an element in a single feature. Therefore for each feature map we would need a set of $C_{in}$ filters of shape $(kernelsize,kernelsize)$ and if we want $C_{out}$ feature maps we would then need $C_{out}$ sets of $(C_{in},kernelsize,kernelsize)$ and $C_{out}$ biases (1 bias for each output feature map) hence the convolution filter's weight is of the shape $(C_{out},C_{in},kernelsize,kernelsize)$ and bias is $C_{out}$

## Types of convolution filter

In the broader context of image processing convolution filters fall into two major categories.

#### 1. Feature Engineered Convolution Filters

Feature engineered convolution filters are hardcoded predefined weights that are designed to extract specific structural attributes from an image. These are traditionally used as independent preprocessing steps to prepare raw data before feeding it into a downstream classifier or machine learning model.

#### 2. Trainable Weights
Trainable filters are the core components of Convolutional Neural Networks (CNNs). Instead of a human engineer manually calculating and defining the weights to extract specific patterns, the weights are initialized with random values and optimized automatically.

Most of the CNN image based models use feature engineered convolution filters to preprocess data before fitting their machine learning models on them

## Understanding how convolution operation is implemented in pytorch

The most basic way to implement a convolution operation is to allow for the convolution filters to slide over the image and generate the feature maps. This is a very slow process. Current day models span 100s of layers where multiple convolution filters are used using this sliding window approach will take enormous amount of time. We will look at how pytorch implements it. Pytorch way of doing things is to make every operation turn out to be some form of matrix multiplication as their gpus are optimised to perform matrix multiplications

We will use `torch.nn.functional.unfold` method -> is a method used to extract sliding windows from a batched input tensor. It takes a multidimensional tensor and "flattens" the spatial neighborhoods into columns which is why this operation is also commonly known as **im2col** (image-to-column). (here convolution operation (product of the image's receptive field with the convolution filter has not taken place yet))

Below is an implementation of unfolding operation for better understanding of it. We are going to extract 2x2 receptive field with a stride of 1, dilation of 1 and padding 0 from two tensors of shape (1, 3, 3) and (3, 3, 3) to understand more carefully how the function works

In [6]:
import math
import torch

# Single channel 3x3 tensor
single_channel_tensor = torch.arange(0, 9, dtype=torch.float32).reshape(1, 3, 3)
print("Single Channel Tensor:")
print(single_channel_tensor)
print(f"Tensor Shape: {single_channel_tensor.shape}")
# Extracting 2x2 receptive fields from the single tensor with stride 2, padding 0 and dilation 1
single_unfolded_tensor = torch.nn.functional.unfold(single_channel_tensor, kernel_size=2, padding=0, stride=1, dilation=1)
print("Unfolded Single Channel Tensor:")
print(single_unfolded_tensor)
print(f"Tensor Shape: {single_unfolded_tensor.shape}")
print("We can see that each 2x2 receptive field has been converted into a column and all receptive fields are stacked horizontally")
print('-'*60)
# 3 channel 3x3 tensor
three_channel_tensor = torch.arange(0, 27, dtype=torch.float32).reshape(3, 3, 3)
print("Three Channel Tensor:")
print(three_channel_tensor)
print(f"Tensor Shape: {three_channel_tensor.shape}")
three_channel_unfolded_tensor = torch.nn.functional.unfold(three_channel_tensor, kernel_size=2, padding=0, stride=1, dilation=1)
print("Unfolded Single Channel Tensor:")
print(three_channel_unfolded_tensor)
print(f"Tensor Shape: {three_channel_unfolded_tensor.shape}")
print(f"Tensor Shape: {three_channel_unfolded_tensor.shape}")
print("We can see that each 2x2 receptive field for each channel has been converted into a column and all receptive fields per channel has been stacked horizontally. All such channel blocks have been stacked vertically.")

Single Channel Tensor:
tensor([[[0., 1., 2.],
         [3., 4., 5.],
         [6., 7., 8.]]])
Tensor Shape: torch.Size([1, 3, 3])
Unfolded Single Channel Tensor:
tensor([[0., 1., 3., 4.],
        [1., 2., 4., 5.],
        [3., 4., 6., 7.],
        [4., 5., 7., 8.]])
Tensor Shape: torch.Size([4, 4])
We can see that each 2x2 receptive field has been converted into a column and all receptive fields are stacked horizontally
------------------------------------------------------------
Three Channel Tensor:
tensor([[[ 0.,  1.,  2.],
         [ 3.,  4.,  5.],
         [ 6.,  7.,  8.]],

        [[ 9., 10., 11.],
         [12., 13., 14.],
         [15., 16., 17.]],

        [[18., 19., 20.],
         [21., 22., 23.],
         [24., 25., 26.]]])
Tensor Shape: torch.Size([3, 3, 3])
Unfolded Single Channel Tensor:
tensor([[ 0.,  1.,  3.,  4.],
        [ 1.,  2.,  4.,  5.],
        [ 3.,  4.,  6.,  7.],
        [ 4.,  5.,  7.,  8.],
        [ 9., 10., 12., 13.],
        [10., 11., 13., 14.],
     

we know that convolution filter's tensor shape is $(C_{out},C_{in},kernelsize,kernelsize)$ we need to reshape the initialized filter such that we have it's column dimension = the unfolded tensor's row dimension to understand how this done have a look at the code below for filters for both single and 3 channel tensors

In [7]:
# Initialized filter
single_channel_filter = torch.randn(size=(1, 1, 2, 2), dtype=torch.float32)
three_channel_filter = torch.randn(size=(3, 3, 2, 2), dtype=torch.float32)
print("Single Channel filter:")
print(single_channel_filter)
print(f"Single Channel Filter Shape: {single_channel_filter.shape}")
print("Three Channel filter:")
print(three_channel_filter)
print(f"Three Channel Filter Shape: {three_channel_filter.shape}")
print("-" * 60)
# Unpack the weights
single_channel_unpacked = single_channel_filter.reshape(1, -1)
three_channel_unpacked = three_channel_filter.reshape(3, -1)
print(f"Single Channel Unpacked Filter Shape: {single_channel_unpacked.shape}")
print(f"Three Channel Unpacked Filter Shape: {three_channel_unpacked.shape}")

Single Channel filter:
tensor([[[[ 0.4903,  2.6355],
          [-0.1841,  1.0143]]]])
Single Channel Filter Shape: torch.Size([1, 1, 2, 2])
Three Channel filter:
tensor([[[[ 0.4948, -0.4480],
          [-1.7829,  1.8560]],

         [[ 1.2381,  1.4688],
          [-0.2214,  0.6612]],

         [[-0.4869, -0.9504],
          [-2.4015, -0.1567]]],


        [[[-0.2973, -0.6539],
          [ 0.0621,  1.3611]],

         [[ 0.5875,  1.5220],
          [ 0.5960,  1.0345]],

         [[ 0.2930,  1.8403],
          [-0.3478, -0.7890]]],


        [[[-0.4188,  0.4317],
          [ 2.0106,  0.4534]],

         [[-0.8436, -0.3435],
          [ 1.2245,  0.2104]],

         [[-0.5847,  1.7756],
          [ 0.7629, -0.0770]]]])
Three Channel Filter Shape: torch.Size([3, 3, 2, 2])
------------------------------------------------------------
Single Channel Unpacked Filter Shape: torch.Size([1, 4])
Three Channel Unpacked Filter Shape: torch.Size([3, 12])


We can see that the column dimension of the unpacked filters is equal to the row dimension of unfolded tensors. Now we perform a matrix multiplication 

In [8]:
single_channel_matrix_product = torch.matmul(single_channel_unpacked, single_unfolded_tensor)
three_channel_matrix_product = torch.matmul(three_channel_unpacked, three_channel_unfolded_tensor)
print(f"Single Channel Matrix Product Shape: {single_channel_matrix_product.shape}")
print(f"Three Channel Matrix Product Shape: {three_channel_matrix_product.shape}")

Single Channel Matrix Product Shape: torch.Size([1, 4])
Three Channel Matrix Product Shape: torch.Size([3, 4])


Now all we need to do is reshape the matrix product into $(N,C_{out},H_{out},W_{out})$ to get the feature maps.

In [9]:
single_channel_h_out = math.floor((3 + (2 * 0) - (1 * (2 - 1)) - 1) / 1) + 1
single_channel_w_out = math.floor((3 + (2 * 0) - (1 * (2 - 1)) - 1) / 1) + 1
three_channel_h_out = math.floor((3 + (2 * 0) - (1 * (2 - 1)) - 1) / 1) + 1
three_channel_w_out = math.floor((3 + (2 * 0) - (1 * (2 - 1)) - 1) / 1) + 1


single_channel_feature_map = single_channel_matrix_product.reshape(1, 1, single_channel_h_out, single_channel_w_out)
three_channel_feature_map = three_channel_matrix_product.reshape(1, 3, three_channel_h_out, three_channel_w_out)

print("Single Channel Feature Map")
print(single_channel_feature_map)
print(f"Single Channel Feature Map: {single_channel_feature_map.shape}")
print("Three Channel Feature Map")
print(three_channel_feature_map)
print(f"Three Channel Feature Map: {three_channel_feature_map.shape}")

Single Channel Feature Map
tensor([[[[ 6.1403, 10.0963],
          [18.0082, 21.9641]]]])
Single Channel Feature Map: torch.Size([1, 1, 2, 2])
Three Channel Feature Map
tensor([[[[-47.3011, -48.0297],
          [-49.4869, -50.2155]],

         [[ 61.6654,  66.8741],
          [ 77.2913,  82.4999]],

         [[ 52.2169,  56.8184],
          [ 66.0213,  70.6228]]]])
Three Channel Feature Map: torch.Size([1, 3, 2, 2])


**This is how Convolution for 2 dimensional tensors (images) is implemented in pytorch!**

The entire implementation to replicate `torch.nn.Conv2d` is written here `pytorch-fundamentals/src/pytorch_fundamentals/layers/cnn.py`

## Comparing implementation of convolution 2d layer with pytorch's version


We will use the same input tensor of shape of an actual 224, 224 RGB image and will compare the convolution implementation with pytorch's implementation

In [10]:
torch.manual_seed(42)
from pytorch_fundamentals.layers.cnn import Conv2dCustom

# Instantiate an input tensor
input_tensor = torch.randn((1, 3, 224, 224), dtype=torch.float32)

# Instantiate pytorch and custom implementation of Convolution Neural Network Layer 
custom_implementation_with_bias = Conv2dCustom(in_channels=3, out_channels=3, kernel_size=2, stride=2, padding=0, dilation=1, bias=True)
torch_implementation_with_bias = torch.nn.Conv2d(in_channels=3, out_channels=3, kernel_size=2, stride=2, padding=0, dilation=1, bias=True)

custom_implementation_without_bias = Conv2dCustom(in_channels=3, out_channels=3, kernel_size=2, stride=2, padding=0, dilation=1, bias=False)
torch_implementation_without_bias = torch.nn.Conv2d(in_channels=3, out_channels=3, kernel_size=2, stride=2, padding=0, dilation=1, bias=False)


# For comparison copying the weights and bias from custom implementation to torch's implementation
with torch.inference_mode():
    torch_implementation_with_bias.weight = custom_implementation_with_bias.weight
    torch_implementation_without_bias.weight = custom_implementation_without_bias.weight

    torch_implementation_with_bias.bias = custom_implementation_with_bias.bias
    torch_implementation_without_bias.bias = custom_implementation_without_bias.bias

# Propagating the input tensor through the implementations and comparing them using torch.allclose() functio
custom_output_tensor_with_bias = custom_implementation_with_bias.forward(input_tensor)
torch_output_tensor_with_bias = torch_implementation_with_bias.forward(input_tensor)
print(f"Custom implementation matches pytorch's implementation (for the case where bias has been added)? {torch.allclose(custom_output_tensor_with_bias, torch_output_tensor_with_bias, atol=1e-6)}")

custom_output_tensor_without_bias = custom_implementation_without_bias.forward(input_tensor)
torch_output_tensor_without_bias = torch_implementation_without_bias.forward(input_tensor)
print(f"Custom implementation matches pytorch's implementation (for the chase where bias has not been added)? {torch.allclose(custom_output_tensor_without_bias, torch_output_tensor_without_bias, atol=1e-6)}")

Custom implementation matches pytorch's implementation (for the case where bias has been added)? True
Custom implementation matches pytorch's implementation (for the chase where bias has not been added)? True


This will set up a very strong foundation for understanding how computer vision models are created